In [1]:
import pandas as pd
import numpy as np
import os
os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\All data')

In [2]:
# Read the column mapping
mapping = pd.read_csv('OBIS columns.csv', sep=';')

print(f"Total columns to create: {len(mapping)}")
print(f"\nColumn mapping:")
print(mapping.head(10))

Total columns to create: 64

Column mapping:
                New file              OBIS
0                     ID               NaN
1               sourceID                id
2             verbatimID      occurrenceID
3         sourceDatabase              OBIS
4             datasetKey        dataset_id
5           datasetTitle       datasetName
6                  issue               NaN
7   decimalLatitudeStart   decimalLatitude
8  decimalLongitudeStart  decimalLongitude
9    decimalLatitudeStop               NaN


In [3]:
# Read the OBIS parquet file
obis_df = pd.read_parquet('../OBIS/obis_subset_merged_FP.parquet')

print(f"OBIS data shape: {obis_df.shape}")
print(f"Number of records: {len(obis_df):,}")

OBIS data shape: (376480, 229)
Number of records: 376,480


In [4]:
# Create the new dataframe with mapped columns
all_merged = pd.DataFrame(index=obis_df.index)

for idx, row in mapping.iterrows():
    new_col = row['New file']
    obis_col = row['OBIS']
    
    # Handle special case: sourceDatabase should be filled with literal "OBIS"
    if new_col == 'sourceDatabase':
        all_merged[new_col] = 'OBIS'
    
    # Handle empty mapping - create empty column
    elif pd.isna(obis_col) or obis_col.strip() == '':
        all_merged[new_col] = None
    
    # Handle multiple source columns (use first available)
    elif ',' in obis_col:
        sources = [c.strip() for c in obis_col.split(',')]
        # Use the first existing source column
        for source in sources:
            if source in obis_df.columns:
                all_merged[new_col] = obis_df[source]
                break
        else:
            # None of the sources exist, create empty column
            all_merged[new_col] = None
    
    # Handle single source column
    else:
        obis_col = obis_col.strip()
        if obis_col in obis_df.columns:
            all_merged[new_col] = obis_df[obis_col]
        else:
            # Source column doesn't exist, create empty column
            all_merged[new_col] = None

print(f"Created dataframe with {len(all_merged.columns)} columns and {len(all_merged):,} rows")
print(f"\nColumn names:")
print(all_merged.columns.tolist())

Created dataframe with 64 columns and 376,480 rows

Column names:
['ID', 'sourceID', 'verbatimID', 'sourceDatabase', 'datasetKey', 'datasetTitle', 'issue', 'decimalLatitudeStart', 'decimalLongitudeStart', 'decimalLatitudeStop', 'decimalLongitudeStop', 'footprintWKT', 'coordinateUncertaintyInMeters', 'hasGeoSpatialIssues', 'depthA', 'depthAType', 'depthB', 'depthBType', 'depthAccuracy', 'gebcoDepth', 'eventID', 'parentEventID', 'waterBody', 'largeMarineEcosystem', 'locationID', 'eventDate', 'startDayOfYear', 'endDayOfYear', 'year', 'eventRemarks', 'verbatimScientificName', 'acceptedScientificName', 'identificationQualifier', 'lifeStage', 'sex', 'taxonomicStatus', 'taxonomicIssue', 'taxonRank', 'phylum', 'class', 'order', 'family', 'genus', 'specificEpithet', 'validAuthority', 'aphiaId', 'acceptedAphiaId', 'individualCount', 'biomass', 'dryMass', 'dynamicProperties', 'basisOfRecord', 'occurrenceRemarks', 'gear', 'gearCategory', 'gearDescription', 'gearDimensions', 'meshSize', 'sieveFract

In [5]:
# Verify the data
print("Sample of the merged data:")
print(all_merged.head())

print(f"\nData types:")
print(all_merged.dtypes)

print(f"\nNull counts:")
null_counts = all_merged.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False))

Sample of the merged data:
     ID                              sourceID       verbatimID sourceDatabase  \
0  None  0001e120-8cd2-45ee-81cb-ceb0c75bd884       I2_879_5_5           OBIS   
1  None  0005af2e-e462-445d-b683-4b980872dc52  72087112_102199           OBIS   
2  None  0006b951-6222-40ff-8c67-a705388c321a  48104054_158359           OBIS   
3  None  000781e4-b51c-4e25-a51c-2e4ad7be88cd  48104061_118394           OBIS   
4  None  000d1695-af9b-4e7b-a65f-6a7eb28f9d6e  62108072_158358           OBIS   

                             datasetKey  \
0  cd1f299a-929a-4a51-b59e-bd5417d65d9c   
1  f7e3baa2-48c9-47d9-9219-0a457615d610   
2  f7e3baa2-48c9-47d9-9219-0a457615d610   
3  f7e3baa2-48c9-47d9-9219-0a457615d610   
4  f7e3baa2-48c9-47d9-9219-0a457615d610   

                                        datasetTitle issue  \
0  North Atlantic and Arctic Isopoda sampled duri...  None   
1  DFO Central and Arctic Multi-species Stock Ass...  None   
2  DFO Central and Arctic Multi-species S

In [6]:
# Save to parquet file
output_path = 'all_merged.parquet'
all_merged.to_parquet(output_path, index=False)

print(f"Successfully saved to {output_path}")
print(f"File contains {len(all_merged):,} records with {len(all_merged.columns)} columns")

Successfully saved to all_merged.parquet
File contains 376,480 records with 64 columns
